## Do not change the code in the cell below ##

In [ ]:
# The pip install can take a minute
%pip install -q urllib3<2.0 datascience ipywidgets
import pyodide_http
pyodide_http.patch_all()
import datascience
from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
import re
def read_file(path):
    """Read a text file and squash all whitespace down to single spaces."""
    with open(path, encoding='utf-8') as f:
        return re.sub(r'\s+', ' ', f.read())

**Note:** In this lecture there is a lot of code. You are not expected to know any of this yet. This is just a preview of the things you will see in the next few weeks.


---

## This is a Jupyter Notebook

A Jupyter Notebook is a data-science environment that combines:

1. **Narrative:** The text describing your analysis
2. **Code:** The program that does the analysis
3. **Results:** The output of the program

The Jupyter environment was created by faculty members at University of California, Berkeley (Fernando Perez). These ideas are now in a lot of different technologies (e.g., Google Colab).


## Our first example: analyzing the text of popular books

We can use the tools of data science to study text.  For example, here we will do some basic analysis of *["The Picture of Dorian Gray"](https://en.wikipedia.org/wiki/The_Picture_of_Dorian_Gray)* (by Oscar Wilde) and of *["A Tale of Two Cities"](https://en.wikipedia.org/wiki/A_Tale_of_Two_Cities)* (by Charles Dickens).

Often the first step in data science is getting the data. Both books come from [Project Gutenberg](https://www.gutenberg.org), and the text files sit in the same folder as this notebook. The `read_file` **function** in the setup cell above opens one and tidies up its spacing. We will talk more about functions later on!


Each book is one long piece of text, so we split it into chapters. The final chapter has Project Gutenberg's licence notice attached to the end of it, so we trim that off.

In [ ]:
dorian_gray_text = read_file('dorian_gray.txt')
dorian_gray_chapters = dorian_gray_text.split('CHAPTER ')[21:]

# Remove the licence notice stuck on the end of the last chapter
dorian_gray_chapters[-1] = dorian_gray_chapters[-1].split('*** END OF')[0]

len(dorian_gray_chapters)

In [ ]:
tale_of_two_cities_text = read_file('tale_of_two_cities.txt')
tale_of_two_cities_chapters = tale_of_two_cities_text.split('CHAPTER ')[46:]

# Remove the licence notice stuck on the end of the last chapter
tale_of_two_cities_chapters[-1] = tale_of_two_cities_chapters[-1].split('*** END OF')[0]

len(tale_of_two_cities_chapters)

Let's look at the start of the first chapter of *The Picture of Dorian Gray*:

In [ ]:
dorian_gray_chapters[0][:600]

## Tables

- A lot of data science is about transforming data. This is often in service of producing **tables**, a widely used data structure that lets us analyse our data more easily.
- In this class you will use the `datascience` library (created specifically for this course!) to manipulate data.


In [ ]:
import datascience
datascience.__version__

In [ ]:
from datascience import *

In [ ]:
Table().with_column('Chapters', dorian_gray_chapters)

## We will learn to summarize data

We will explore data by extracting summaries. For example, we might ask how often each character appears in each chapter. We can use snippets of code to answer these questions.

In [ ]:
np.char.count(dorian_gray_chapters, 'Dorian')

In [ ]:
np.char.count(dorian_gray_chapters, 'Henry')

In [ ]:
np.char.count(dorian_gray_chapters, 'Basil')

We can convert the results of our analysis into more tables.

In [ ]:
counts = Table().with_columns([
    'Dorian', np.char.count(dorian_gray_chapters, 'Dorian'),
    'Henry', np.char.count(dorian_gray_chapters, 'Henry'),
    'Basil', np.char.count(dorian_gray_chapters, 'Basil'),
])
counts

## We will learn to visualize data

- How many times is each character mentioned in Chapter 1, how many times in Chapters 1 and 2, and so on?
- As we saw above, we could answer this with a table, but there are a lot of chapters! Let's try something else.


In [ ]:
cum_counts_dorian = np.cumsum(counts.column("Dorian"))
cum_counts_henry = np.cumsum(counts.column("Henry"))
cum_counts_basil = np.cumsum(counts.column("Basil"))

cumulative_table = Table().with_columns(
    'Chapter', np.arange(1, 21, 1),
    'Dorian', cum_counts_dorian,
    'Henry', cum_counts_henry,
    'Basil', cum_counts_basil
)

cumulative_table.plot(column_for_xticks='Chapter')
plots.title('Cumulative Number of Times Name Appears')
plots.show()

Here, we have plotted what we call *cumulative counts*.

What can we tell from this visualization?  What questions does this raise about the roles of Dorian, Henry and Basil in the book?

In [ ]:
# The chapters of A Tale of Two Cities
Table().with_column('Chapters', tale_of_two_cities_chapters)

We can explore the characters in *A Tale of Two Cities* using the same kind of analysis.

In [ ]:
# Counts of names in the chapters of A Tale of Two Cities
counts = Table().with_columns([
    'Charles', np.char.count(tale_of_two_cities_chapters, 'Charles'),
    'Sydney', np.char.count(tale_of_two_cities_chapters, 'Sydney'),
    'Lucie', np.char.count(tale_of_two_cities_chapters, 'Lucie'),
    'Manette', np.char.count(tale_of_two_cities_chapters, 'Manette'),
    'Defarge', np.char.count(tale_of_two_cities_chapters, 'Defarge'),
])
counts

In [ ]:
# Plot the cumulative counts
cumulative_table = Table().with_columns(
    'Chapter', np.arange(1, 46, 1),
    'Charles', np.cumsum(counts.column("Charles")),
    'Sydney', np.cumsum(counts.column("Sydney")),
    'Lucie', np.cumsum(counts.column("Lucie")),
    'Manette', np.cumsum(counts.column("Manette")),
    'Defarge', np.cumsum(counts.column("Defarge")),
)

cumulative_table.plot(column_for_xticks='Chapter')
plots.title('Cumulative Number of Times Names Appear in A Tale of Two Cities')
plots.show()

We can use interactive tools as well!

In [ ]:
Table.interactive_plots()
cumulative_table.plot(column_for_xticks=0)

## Visualizing multiple variables

- How long are the chapters in a book?
- How many sentences are in a chapter? We can count full stops as a rough guide.

You don't need to worry about understanding the code below for today!


In [ ]:
len(dorian_gray_text)

In [ ]:
# In each chapter, count the number of all characters;
# call this the "length" of the chapter.
# Also count the number of periods (full stops).

length_tpdg = Table().with_columns([
    'Length', [len(s) for s in dorian_gray_chapters],
    'Periods', np.char.count(dorian_gray_chapters, '.')
])
length_atotc = Table().with_columns([
    'Length', [len(s) for s in tale_of_two_cities_chapters],
    'Periods', np.char.count(tale_of_two_cities_chapters, '.')
])

In [ ]:
# The counts for The Picture of Dorian Gray
length_tpdg

In [ ]:
# The counts for A Tale of Two Cities
length_atotc

Now that we have a table for each book giving us information on:

- length per chapter
- number of periods per chapter

We might consider examining how these two variables are related. Below is what is called a **scatter plot** (we will talk about this plot in more depth later on!)


In [ ]:
Table.static_plots()
plots.figure(figsize=(10, 10))
plots.scatter(length_tpdg.column('Periods'), length_tpdg.column('Length'), color='darkblue')
plots.scatter(length_atotc.column('Periods'), length_atotc.column('Length'), color='gold')
plots.xlabel('Number of periods in chapter')
plots.ylabel('Number of characters in chapter')
plots.title('Relationship between numbers of characters and periods in a chapter');

This sub-example illustrates the relationship between different facets of our course: namely, the exploration and prediction facets.